In [ ]:
from pathlib import Path
CWD = Path(__name__).resolve().parent

source_pdf = CWD / "test_pdfs/minecraft.pdf"
if not source_pdf.is_file():
    raise ValueError(f"source file at '{source_pdf}' does not exist")

secrets = CWD / "secrets.env"

if not secrets.is_file():
    raise ValueError(f"secrets file at '{secrets}' does not exist")

from dotenv import load_dotenv
load_dotenv(secrets)


### Text Extraction From Source .PDF
We use pymupdf4llm to handle pulling the PDF text into a Markdown format, and then perform manual text processing steps to prepare it for segmentation:
1. **Replace all occurences of single line-breaks with a single whitespace.** (`r"(?<!\n)\n(?!\n)" -> " "`)<br>
    => Academic papers formatted in column layout will break sentences with `\n`
2. **Replace all occurences of 2 or more line-breaks with a single line-break** (`r"(\n{2,})" -> "\n"`)
    => Organizational formatting in the paper may occur as 2+ line-breaks, which is unnecessary for segmentation purposes.
3. **Replace all occurences of 2 or more whitespaces with a single whitespace** (`r"\s{2,}" -> " "`)
    => Formatting artifacts from the source text may include long strings of whitespace, which are unnecessary for segmentation purposes.
4. **Replace all occurences of lines that only consist of a single digit with whitespace** (`r"\n^[0-9]$\n"`)
    => Sentences spanning across pages may be cut off by the occurence of a page number

In [ ]:
from typing import TypedDict
import fitz, pymupdf4llm
import re

class ExtractedSource(TypedDict):
    source: str
    normalized_text: str

def extract_text(pdf_path: Path) -> str:
    
    with fitz.open(pdf_path) as doc:
        md_text = pymupdf4llm.to_markdown(doc, ignore_images=True, ignore_graphics=True, ignore_code=True)

        md_text = re.sub(r"(?<!\n)\n(?!\n)", " ", md_text) # a single line break -> 1 whitespace
        md_text = re.sub(r"\n{2,}", "\n", md_text) # 2+ line breaks -> 1 line break
        md_text = re.sub(r"\s{2,}", " ", md_text) # 2+ whitespaces -> 1 whitespace
        md_text = re.sub(r"\n^[0-9]$\n", " ", md_text, flags=re.MULTILINE) # any lines that are just a single digit (page number)
    
    return md_text

extracted_pdf:ExtractedSource = { "source": source_pdf.name, "normalized_text": extract_text(source_pdf)}
print(f"==SOURCE==\n{extracted_pdf['source']}")
print(f"==TEXT==\n{extracted_pdf['normalized_text']}")

### Source Segmentation into Chunks
We use [Segment-Any-Text (SaT)](https://github.com/segment-any-text/wtpsplit) to divide the normalized source text into semantically-relevant chunks

In [ ]:
SEGMENTER_MODEL = "sat-3l-sm" # SaT model
PARAGRAPH_THRESHOLD = 0.5 # bias for paragraph length
MIN_CHUNK_TOKENS_THRESH = 20 # minimum tokens before a segment is considered a chunk

In [ ]:
# segment source into chunks (embeddings come later)
from typing import TypedDict
from statistics import mean, median
from wtpsplit import SaT
from tokenizers import Tokenizer

TOKEN_MODEL = "gpt2"

class SPOTriple (TypedDict):
    s:str # subject
    p:str # predicate
    o:str # object

class Chunk (TypedDict):
    id:int
    raw_text:str
    approx_n_tokens:int
    embedding:list[float]
    triples:list[SPOTriple]

_tokenizer = Tokenizer.from_pretrained(TOKEN_MODEL)
class TextSegmenter:
    """
    Segments text into chunks using wtpsplit ("Segment Any Text"):
    https://github.com/segment-any-text/wtpsplit
    """
    _model: str | None = None
    _instance: SaT | None = None

    @classmethod
    def configure(cls, model: str) -> None:
        cls._model = model
        cls._instance = None

    @classmethod
    def instance(cls) -> SaT:
        if cls._model is None:
            raise RuntimeError("TextSegmenter model not configured")

        if cls._instance is None:
            cls._instance = SaT(
                cls._model,
                ort_providers=["CPUExecutionProvider"],
            )

        return cls._instance

    @classmethod
    def _normalize_segments(cls, segments: list[str]) -> None:
        """
        mutably collapse 2 segments when the boundary is a dash, sanitize
        """
        out = []
        i = 0
        n = len(segments)

        while i < n:
            s = segments[i]
            if s.endswith('-') and i + 1 < n:
                s = s[:-1] + segments[i + 1]
                i += 2
            else:
                i += 1

            out.append(s)

        segments.clear()
        segments.extend(out)

    @classmethod
    def create_segments(cls, text: str) -> list[str]:

        instance = cls.instance()
        segments = instance.split(text,
            strip_whitespace=True, 
            remove_whitespace_before_inference=True, 
            paragraph_threshold=PARAGRAPH_THRESHOLD)
        
        cls._normalize_segments(segments)

        return segments

def process_chunk_text(chunk_text:str) -> Chunk:
    """ compute token statistics & trim chunk list by minimum token length"""
    chunk_text = chunk_text.strip()
    enc = _tokenizer.encode(chunk_text)
    raw_chunk = {
        "id": hash(chunk_text),
        "raw_text": chunk_text,
        "approx_n_tokens": len(enc),
    }
    return raw_chunk

TextSegmenter.configure(model=SEGMENTER_MODEL)
chunk_texts:list[str] = TextSegmenter.create_segments(extracted_pdf['normalized_text'])
if not chunk_texts:
    raise Exception(f"source '{extracted_pdf['source']}' did not generate chunks")

from typing import Hsh
# assemble raw chunk list & compute token stats
token_counts = []
min_tokens, max_tokens = float('inf'), 0
chunks:list[Chunk] = []
for text in chunk_texts:
    chunk = process_chunk_text(text)
    n_tokens = chunk['approx_n_tokens']

    if not chunk or n_tokens < MIN_CHUNK_TOKENS_THRESH:
        continue # skip empty and small chunks
    if chunk['id'] in [c.id for c in chunks]:
        continue # skip duplicate chunks

    chunks.append(chunk)

    # token stats tracking
    token_counts.append(n_tokens)
    if(n_tokens < min_tokens):
        min_tokens = n_tokens
    if(n_tokens > max_tokens):
        max_tokens = n_tokens

print(f"\n====TOKEN STATS FOR SOURCE '{source_pdf}'====")
print(f"# Chunks: {len(token_counts)}")
print(f"Median chunk tokens: {median(token_counts)}")
print(f"Mean chunk tokens: {mean(token_counts)}")
print(f"Min chunk token count: {min_tokens}")
print(f"Max chunk token count: {max_tokens}")

### Generate chunk embeddings
We create batches of requests for an embedding model, and generate the embeddings for the raw chunk text

In [ ]:
# batch generate embeddings for raw chunk text
import litellm
from asyncio import Semaphore, Task, create_task, gather

EMBED_MODEL = "bedrock/amazon.titan-embed-text-v2:0"
MAX_PARALLEL_EMBED = 10 # maximum concurrent embedding model requests
BATCH_TOKEN_TARGET = 300 # pack chunks into a batch until we cross this threshold
MAX_BATCH_ITEMS = 64 # cap on request size

_embed_sem = Semaphore(MAX_PARALLEL_EMBED)

async def _embed_batch(chunks:list[Chunk]) -> list[list[float]]:
    async with _embed_sem:
        resp = await litellm.aembedding(model=EMBED_MODEL, input=[c["raw_text"] for c in chunks])
    data = resp['data']
    if len(data) != len(chunks):
        raise RuntimeError("Embedding count mismatch.. cant map embeddings to owning chunks")
    
    # NOTE: order is preserved such that the embedding at output[i] corresponds to the chunk at chunks[i]
    return [item['embedding'] for item in data]


async def _embed_and_update_batch(batch:list[Chunk]) -> None:
    # get embedding for batch
    embs = await _embed_batch(batch)

    for chunk, embedding in zip(batch, embs, strict=True):
        chunk["embedding"] = embedding

def _make_batches(chunks:list[Chunk]) -> list[list[Chunk]]:
    batches: list[list[Chunk]] = []
    i = 0
    n = len(chunks)

    while i < n :
        batch: list[Chunk] = []
        tok = 0

        while i < n and len(batch) < MAX_BATCH_ITEMS and tok < BATCH_TOKEN_TARGET:
            batch.append(chunks[i])
            tok += chunks[i]['approx_n_tokens']
            i+=1
        
        # if starting with a chunk that crosses target then batch will contain just that chunk
        if not batch:
            batch = [chunks[i]]
            i += 1
        
        batches.append(batch)

    return batches

async def generate_chunk_embeddings(chunks:list[Chunk])->None:

    # schedules embedding request coroutines in waves (avoid unbounded scheduling)
    batches = _make_batches(chunks)
    in_flight: list[Task[None]] = []
    WAVE = MAX_PARALLEL_EMBED * 2

    for batch in batches:
        in_flight.append(create_task(_embed_and_update_batch(batch)))
        if len(in_flight) >= WAVE:
            await gather(*in_flight)
            in_flight.clear()

    if in_flight:
        await gather(*in_flight)


await generate_chunk_embeddings(chunks)

In [19]:
# helper - print small view into the chunk objects
def print_chunks_view(_max_chunks_print = 10, _max_field_len_print = 50):
    def _ellipsize(s: str, max_len: int) -> str:
        s = str(s)
        return s if len(s) <= max_len else s[:max_len - 3] + f"...(+{len(s[max_len-3:])})"
    
    print("====CHUNKS (VIEW)====")
    for i in range(min(len(chunks), _max_chunks_print)): # print up to 10
        _chunk = chunks[i]
        print(f"id (#{i+1}):\t{_ellipsize(_chunk['id'], _max_field_len_print)}")
        print(f"text:\t\t{_ellipsize(_chunk['raw_text'], _max_field_len_print)}")
        print(f"tokens:\t\t{_ellipsize(_chunk['approx_n_tokens'], _max_field_len_print)}")
        if 'embedding' in _chunk.keys():
            print(f"embed:\t\t{_ellipsize(_chunk['embedding'], _max_field_len_print)}")
        if 'triples' in _chunk.keys():
            print(f"trips:\t\t{_ellipsize(_chunk['triples'], _max_field_len_print)}")
        print("\n")

print_chunks_view()

====CHUNKS (VIEW)====
id (#1):	-8404635112451787551
text:		# **Detecting Aimbot Usage in Minecraft Using a...(+35)
tokens:		20
embed:		[-0.018900137394666672, 0.05828872323036194, 0....(+22675)
trips:		[{'s': 'Detecting Aimbot Usage', 'p': 'in', 'o'...(+101)


id (#2):	436694556930953333
text:		The gaming industry has long been at the cuttin...(+1707)
tokens:		340
embed:		[-0.008361529558897018, 0.027321437373757362, 0...(+22675)
trips:		[{'s': 'gaming industry', 'p': 'has been at', '...(+1658)


id (#3):	4502777806819250988
text:		**Keywords:** Cheat Detection, Machine Learning...(+33)
tokens:		22
embed:		[-0.021504323929548264, 0.06631588190793991, 0....(+22654)
trips:		[{'s': 'Cheat Detection', 'p': 'is a', 'o': 'pr...(+248)


id (#4):	2645550369675323464
text:		Historically the video game industry has stayed...(+1526)
tokens:		316
embed:		[0.008595090359449387, -0.002683141501620412, 0...(+22672)
trips:		[{'s': 'video game industry', 'p': 'has stayed ...(+1565)


id (#5):	-64028291

### Subject-Predicate-Object (SPO) Triple Extraction
We extract SPO triples from each chunk.

In [20]:
import dspy
import json

EXTRACTION_MODEL = "bedrock/us.amazon.nova-pro-v1:0"
MAX_PARALLEL_EXTRACT = 8 # maximum extraction calls to send concurrently

triple_sem = Semaphore(MAX_PARALLEL_EXTRACT)

class _ExtractTriples(dspy.Signature):
    """
    Extract subject predicate object triples from the source text.
    Be thorough, accurate, and faithful to the source text.
    Return a JSON array under 'triples_json' like:
    [
      {"subject": "...", "predicate": "...", "object": "..."},
      ...
    ]
    """
    source_text = dspy.InputField()
    triples_json = dspy.OutputField(desc="JSON array of {subject, predicate, object}")


class TripleExtractor:
    """
    Async triplet extractor from chunk text
    """

    def __init__(self, model: str) -> None:
        self._lm = dspy.LM(model)

    async def extract(self, text: str) -> list[dict[str, str]]:
        # Async DSPy call within a per-task context to avoid global settings
        try:
            with dspy.context(lm=self._lm):
                pred = await dspy.Predict(_ExtractTriples).acall(source_text=text)
        except Exception as e:
            print(f"Triplet extraction error: {type(e).__name__}: {e}")
            return []

        raw = getattr(pred, "triples_json", "") or "[]"

        try:
            data = json.loads(raw)
            if not isinstance(data, list):
                raise ValueError("triples_json must be a JSON array")
        except Exception as e:
            print(f"DSPy triples_json parse failure: {e}; raw={raw[:200]!r}")
            return []

        out: list[SPOTriple] = []
        for item in data:
            if not isinstance(item, dict):
                continue
            s = (item.get("subject") or "").strip()
            p = (item.get("predicate") or "").strip()
            o = (item.get("object") or "").strip()

            if not (s and p and o):
                continue # skip empty / broken triples

            out.append({"s": s, "p": p, "o": o}) # SPO Triple

        return out
    

triple_extractor = TripleExtractor(model=EXTRACTION_MODEL)

async def _extract_triple_from_chunk(chunk: Chunk) -> None:
    async with triple_sem:
        triples = await triple_extractor.extract(chunk['raw_text'])

    chunk['triples'] = triples
    
async def extract_chunk_triples(chunks:list[Chunk]) -> None:
    # schedule extraction tasks in waves
    WAVE = MAX_PARALLEL_EXTRACT * 2
    
    in_flight: list[Task[None]] = []

    for chunk in chunks:
        in_flight.append(create_task(_extract_triple_from_chunk(chunk)))
        if len(in_flight) >= WAVE:
            await gather(*in_flight)
            in_flight.clear()

    if in_flight:
        await gather(*in_flight)

await extract_chunk_triples(chunks)
print_chunks_view()

====CHUNKS (VIEW)====
id (#1):	-8404635112451787551
text:		# **Detecting Aimbot Usage in Minecraft Using a...(+35)
tokens:		20
embed:		[-0.018900137394666672, 0.05828872323036194, 0....(+22675)
trips:		[{'s': 'Detecting Aimbot Usage', 'p': 'in', 'o'...(+101)


id (#2):	436694556930953333
text:		The gaming industry has long been at the cuttin...(+1707)
tokens:		340
embed:		[-0.008361529558897018, 0.027321437373757362, 0...(+22675)
trips:		[{'s': 'gaming industry', 'p': 'has been at', '...(+1658)


id (#3):	4502777806819250988
text:		**Keywords:** Cheat Detection, Machine Learning...(+33)
tokens:		22
embed:		[-0.021504323929548264, 0.06631588190793991, 0....(+22654)
trips:		[{'s': 'Cheat Detection', 'p': 'is a', 'o': 'pr...(+248)


id (#4):	2645550369675323464
text:		Historically the video game industry has stayed...(+1526)
tokens:		316
embed:		[0.008595090359449387, -0.002683141501620412, 0...(+22672)
trips:		[{'s': 'video game industry', 'p': 'has stayed ...(+1565)


id (#5):	-64028291

In [27]:
OUT_FILE_NAME = "result.json"

with open(OUT_FILE_NAME, "w") as out_file:
    out_file.write("[")

    # single source for now
    out_file.write("{ "+f'"source":"{source_pdf.name}",\n')
    out_file.write('"chunks": [\n')

    for i,chunk in enumerate(chunks):
        json.dump(chunk, out_file, ensure_ascii=False)
        if i != len(chunks)-1:
            out_file.write(",\n")
        out_file.flush()

    out_file.write("\n]\n}")
    out_file.write("]")